# 8. Error pictures of single scenes (CPU, camera layer only)

Numbers say THAT a scene went wrong; pictures say HOW. For each chosen scene and each model: a table with one line
per frame, the camera image next to predicted depth, LiDAR ground truth and the signed error, and the driven path
from above. Everything comes from SAVED predictions and SAVED ground truth; only the camera images are downloaded.
A scene cannot be fetched alone, so each scene costs the download of its whole block: 2 to 5 minutes per block,
about 20 minutes for the six scenes here. No GPU. Pictures are also saved under `figures/` in persistent storage.

*In the committed copy the embedded pictures were removed to keep the file small. Selected pictures are in
`results/figures/`; running the notebook draws all of them again.*

In [1]:
# --- 1. Configuration ---
PERSIST_MODE = "drive"
DRIVE_ROOT = "/content/drive/MyDrive/vggt-omega-aura-benchmark"   # where predictions, ground truth and results live.
# Work already saved there is skipped. To run EVERYTHING again from the images up, name an empty folder here,
# the same one in every notebook of the run. The Hugging Face token is still found in the usual folder's .env.
RUN_TAG = "phase7_front_medium"       # the folder of this run in persistent storage. A name from the time of the work:
                                      # every notebook of the run must use the same one, the results live under it
CAMERA = "front_medium"
MODELS = ["vggt_omega_512", "vggt_1b"]
SCENES = [   # (split, block, scene id, why this scene)
    ("val", 10, "2026-01-08-15-27-06|43", "depth outlier: AbsRel 1.64 although the pose is good"),
    ("train", 13, "2026-01-08-16-15-15|16", "wet night: direction of travel predicted reversed (161 deg)"),
    ("train", 12, "2026-01-08-16-15-15|36", "wet night: direction reversed, pose scale 4.7 times the depth scale"),
    ("train", 82, "2025-06-20-10-08-06|89", "dry day, motorway at 12 km/h: pose scale 0.32 of the depth scale"),
    ("test", 3, "2025-08-04-11-19-58|52", "test split, slow urban scene: moving objects 0.447"),
]
ADD_TYPICAL_TEST_SCENE = True         # plus the test scene with the MEDIAN depth error, as the reference for "normal"
ROWS_PER_FIGURE = 4                   # three frames spread over the scene, plus the frame with the largest error

In [ ]:
# === CODE SYNC (auto-generated by `python -m vggt_aura.sync`, do not edit) ===
raise RuntimeError("The sync cell is empty. On your own machine, in the project folder, run:  python -m vggt_aura.sync   and reopen this notebook.")

In [3]:
# --- 3. Session ---
from vggt_aura.session import start_session

session = start_session(persist_mode=PERSIST_MODE, drive_root=DRIVE_ROOT, require_gpu=False)

Mounted at /content/drive
installing vggt_omega
installing pybind11
installing fzi_aura
persist root: /content/drive/MyDrive/vggt-omega-aura-benchmark
data root   : /content/data/fzi-aura (runtime disk, wiped at session end)


In [4]:
# --- 4. Which scenes, and which blocks hold them ---
import pandas as pd
from vggt_aura import aura_data as ad, evaluation as ev, metrics as mt, pipeline as pl

pd.set_option("display.width", 220)
scenes = list(SCENES)
if ADD_TYPICAL_TEST_SCENE:
    rows, table = pl.load_run(session.persist_root, RUN_TAG, MODELS[0])
    overview = ev.scene_overview(rows, table)
    test = overview[overview["split"] == "test"].sort_values("abs_rel").reset_index(drop=True)
    test = test[~test["scene_id"].isin([scene_id for _, _, scene_id, _ in scenes])].reset_index(drop=True)   # not one drawn anyway
    if len(test):
        typical = test.iloc[len(test) // 2]
        scenes.append(("test", int(typical["block"]), str(typical["scene_id"]),
                       f"typical test scene: median depth error ({typical['abs_rel']:.3f})"))
for entry in scenes:
    print(entry)
chunks, scene_blocks, hub_files = ad.fetch_release_tables(session.data_root / "_release_tables")
EXCLUDED = ad.fetch_excluded_scene_ids(session.data_root / "_release_tables")

('val', 10, '2026-01-08-15-27-06|43', 'depth outlier: AbsRel 1.64 although the pose is good')
('train', 13, '2026-01-08-16-15-15|16', 'wet night: direction of travel predicted reversed (161 deg)')
('train', 12, '2026-01-08-16-15-15|36', 'wet night: direction reversed, pose scale 4.7 times the depth scale')
('train', 82, '2025-06-20-10-08-06|89', 'dry day, motorway at 12 km/h: pose scale 0.32 of the depth scale')
('test', 3, '2025-08-04-11-19-58|52', 'test split, slow urban scene: moving objects 0.447')
('test', 1, '2025-08-04-11-19-58|81', 'typical test scene: median depth error (0.076)')


In [5]:
# --- 5. The pictures ---
import shutil
import matplotlib.pyplot as plt
import numpy as np
from fzi_aura import FZIAURADataset
from vggt_aura import figures as fg

out_dir = session.persist_root / "figures" / RUN_TAG
out_dir.mkdir(parents=True, exist_ok=True)
by_block = {}
for split, block, scene_id, why in scenes:
    by_block.setdefault((split, block), []).append((scene_id, why))

for (split, block), wanted in by_block.items():
    print("=" * 110)
    print(pl.block_tag(split, block))
    data_root = session.data_root / f"pictures_{pl.block_tag(split, block)}"
    scene_ids = ad.block_scene_ids(scene_blocks, split, block, EXCLUDED)
    if not ad.block_on_disk(data_root, scene_ids, [pl.CAMERA_LAYER]):
        ad.download_block(data_root, split, block, scene_ids, [pl.CAMERA_LAYER])
    dataset = FZIAURADataset(data_root, split=split)
    for scene_id, why in wanted:
        for model in MODELS:
            bundle = fg.scene_bundle(session.persist_root, dataset, scene_id, CAMERA, pl.ModelRunner(model).folder)
            arrays, truth, labels, scale = bundle["arrays"], bundle["truth"], bundle["labels"], bundle["scale"]
            table = fg.frame_table(arrays["depth"], truth, labels, scale)
            scene_abs_rel = float(np.average(table["abs_rel"], weights=table["lidar_pixels"]))
            print()
            print(f"##### {scene_id} | {model} | {why}")
            print(f"scene AbsRel {scene_abs_rel:.3f} | {bundle['frames']} frames | scale {scale:.2f} m per model unit | "
                  f"frame scale ratio from {table['scale_ratio'].min():.2f} to {table['scale_ratio'].max():.2f}")
            print(table.sort_values("abs_rel", ascending=False).head(6).round(3).to_string(index=False))
            spread = np.linspace(0, bundle["frames"] - 1, ROWS_PER_FIGURE - 1).round().astype(int).tolist()
            chosen = sorted(set(spread + [int(table["abs_rel"].idxmax())]))
            stem = f"{bundle['scene_name']}_{model}"
            figure = fg.scene_figure(bundle["images"], arrays["depth"], truth, labels, scale, chosen,
                                     f"{scene_id} | {model} | AbsRel {scene_abs_rel:.3f} | {why}")
            figure.savefig(out_dir / f"{stem}_depth.png", dpi=80)
            plt.show()
            plt.close(figure)
            figure = fg.trajectory_figure(arrays["extrinsics"], arrays["gt_camera0_from_camera"], f"{scene_id} | {model} | driven path")
            figure.savefig(out_dir / f"{stem}_path.png", dpi=80)
            plt.show()
            plt.close(figure)
    shutil.rmtree(data_root, ignore_errors=True)
print()
print("pictures saved to", out_dir)

val_block000010
  downloading with the toolkit, decompressing with xz on all 2 cores


  fast unpack: {'archives': 2, 'xz_decompressed_on_all_cores': 0, 'download_s': 49.9, 'verify_and_decompress_s': 19.6, 'extract_s': 38.2}

##### 2026-01-08-15-27-06|43 | vggt_omega_512 | depth outlier: AbsRel 1.64 although the pose is good
scene AbsRel 1.640 | 40 frames | scale 137.92 m per model unit | frame scale ratio from 0.89 to 1.11
 frame  lidar_pixels  abs_rel  median_signed_error  scale_ratio  moving_share  abs_rel_moving
    19         20074    4.350               -0.017        1.018         0.028           2.130
    38         17627    3.011               -0.086        1.094         0.029           0.212
    39         15396    2.695               -0.097        1.108         0.033           0.135
    18         21703    2.375               -0.003        1.003         0.016           0.390
     0         27107    2.236                0.118        0.895         0.055           0.204
    37         18844    2.206               -0.064        1.068         0.081           0.211



##### 2026-01-08-15-27-06|43 | vggt_1b | depth outlier: AbsRel 1.64 although the pose is good
scene AbsRel 0.755 | 40 frames | scale 156.57 m per model unit | frame scale ratio from 0.94 to 1.05
 frame  lidar_pixels  abs_rel  median_signed_error  scale_ratio  moving_share  abs_rel_moving
    19         19142    1.420               -0.000        1.000         0.026           0.240
    38         16710    1.277                0.026        0.975         0.025           0.158
    39         14833    1.233               -0.003        1.003         0.031           0.135
    28         17550    1.090               -0.029        1.030         0.071           0.142
    37         18053    1.052                0.018        0.983         0.079           0.147
    29         16521    1.045                0.002        0.998         0.030           0.209


train_block000013
  downloading with the toolkit, decompressing with xz on all 2 cores


  fast unpack: {'archives': 2, 'xz_decompressed_on_all_cores': 0, 'download_s': 63.5, 'verify_and_decompress_s': 47.9, 'extract_s': 94.3}

##### 2026-01-08-16-15-15|16 | vggt_omega_512 | wet night: direction of travel predicted reversed (161 deg)
scene AbsRel 0.139 | 40 frames | scale 319.12 m per model unit | frame scale ratio from 0.94 to 1.09
 frame  lidar_pixels  abs_rel  median_signed_error  scale_ratio  moving_share  abs_rel_moving
     1         29627    0.268                0.027        0.973         0.015           1.880
    10         28597    0.230               -0.066        1.071         0.079           0.660
    28         29323    0.224               -0.055        1.058         0.134           0.672
    29         28654    0.187               -0.084        1.092         0.013           0.533
     0         29216    0.180               -0.071        1.077         0.002           0.279
     2         25651    0.173               -0.013        1.014         0.000           

...


##### 2025-08-04-11-19-58|52 | vggt_1b | test split, slow urban scene: moving objects 0.447
scene AbsRel 0.196 | 40 frames | scale 66.81 m per model unit | frame scale ratio from 0.86 to 1.07
 frame  lidar_pixels  abs_rel  median_signed_error  scale_ratio  moving_share  abs_rel_moving
    36         45375    0.393               -0.059        1.062         0.118           2.054
    35         46611    0.341               -0.058        1.062         0.142           1.377
    37         46069    0.319               -0.058        1.062         0.069           2.442
    38         46930    0.313               -0.064        1.068         0.069           2.166
    34         47682    0.303               -0.058        1.062         0.135           1.160
    32         46978    0.292               -0.058        1.062         0.114           1.249


test_block000001
  downloading with the toolkit, decompressing with xz on all 2 cores


  fast unpack: {'archives': 2, 'xz_decompressed_on_all_cores': 0, 'download_s': 164.4, 'verify_and_decompress_s': 36.2, 'extract_s': 107.3}

##### 2025-08-04-11-19-58|81 | vggt_omega_512 | typical test scene: median depth error (0.076)
scene AbsRel 0.076 | 40 frames | scale 77.72 m per model unit | frame scale ratio from 0.95 to 1.03
 frame  lidar_pixels  abs_rel  median_signed_error  scale_ratio  moving_share  abs_rel_moving
    32         39512    0.124                0.054        0.949         0.031           0.128
    31         39310    0.110                0.022        0.978         0.030           0.113
    27         39932    0.109               -0.031        1.032         0.060           0.108
    26         39780    0.100               -0.014        1.015         0.066           0.176
     0         27353    0.097                0.021        0.980         0.056           0.202
    25         40788    0.095                0.003        0.997         0.066           0.100



##### 2025-08-04-11-19-58|81 | vggt_1b | typical test scene: median depth error (0.076)
scene AbsRel 0.144 | 40 frames | scale 93.13 m per model unit | frame scale ratio from 0.85 to 1.06
 frame  lidar_pixels  abs_rel  median_signed_error  scale_ratio  moving_share  abs_rel_moving
     0         25867    0.215                0.173        0.853         0.054           0.131
     8         35903    0.202                0.061        0.943         0.043           0.182
     6         33916    0.201                0.067        0.937         0.050           0.186
     7         34570    0.193                0.052        0.951         0.039           0.188
     2         26209    0.190                0.129        0.885         0.089           0.141
     1         26199    0.188                0.133        0.882         0.064           0.112



pictures saved to /content/drive/MyDrive/vggt-omega-aura-benchmark/figures/phase7_front_medium
